Step 1: Upload the Excel File & Create raw_data.csv

In [2]:
# ==============================================================================
# CELL 1 / STEP 1: UPLOAD RAW_DATA.CSV & RUN AUTOMATED AUDIT
# ==============================================================================
import pandas as pd
import numpy as np
import io
from google.colab import files

print("Please upload your raw_data.csv file:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df_raw = pd.read_csv(io.BytesIO(uploaded[file_name]))

print("\n" + "="*55)
print(f"RAW DATASET AUDIT: {df_raw.shape[0]} Rows x {df_raw.shape[1]} Columns")
print("="*55)

missing_counts = df_raw.isnull().sum()
print("Columns with missing values:")
print(missing_counts[missing_counts > 0])

Please upload your raw_data.csv file:


Saving raw_data (1).csv to raw_data (1).csv

RAW DATASET AUDIT: 80 Rows x 16 Columns
Columns with missing values:
age_group                             7
discovery_platform                    5
ad_frequency                          7
online_product_category               4
clicked_ad_3m                         8
ad_trust                              4
purchase_influence                    5
ad_ignore_reason                      5
discount_importance                   4
preferred_promotion                   8
choose_promoted_product_likelihood    8
dtype: int64


Step 2: Handle Missing Values, Outliers & Normalization

In [3]:
# ==============================================================================
# CELL 2 / STEP 2: PREPROCESSING (WITHOUT NORMALIZATION)
# ==============================================================================
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

df_clean = df_raw.copy()

# ------------------------------------------------------------------------------
# 2-A. TEXT SANITIZATION & BALANCED ORDINAL ENCODING (1 TO 5 SCALE)
# ------------------------------------------------------------------------------
print("--- [STAGE 1] TEXT CLEANING & 1-5 ORDINAL ENCODING ---")

# Replace any typographical dashes with a standard hyphen
df_clean['age_group'] = df_clean['age_group'].str.replace(r'[\u2013\u2014\ufffd]', '-', regex=True)

# Map ordinal features to 1-5 scale (identical to Likert survey scales)
age_rank = {'Under 18': 1, '18-22': 2, '23-30': 3, '31-45': 4, 'Above 45': 5}
freq_rank = {'Never': 1, 'Rarely': 2, 'Sometimes': 3, 'Often': 4, 'Very often': 5}

df_clean['age_group_ord'] = df_clean['age_group'].map(age_rank)
df_clean['ad_frequency_ord'] = df_clean['ad_frequency'].map(freq_rank)
print("  ✓ Converted demographics to balanced 1–5 ordinal scales.")

# ------------------------------------------------------------------------------
# 2-B. MULTI-DIMENSIONAL KNN IMPUTATION (BALANCED 5D METRIC SPACE)
# ------------------------------------------------------------------------------
print("\n--- [STAGE 2] 5D DISTANCE-WEIGHTED KNN IMPUTATION ---")

knn_features = [
    'age_group_ord', 'ad_frequency_ord',
    'ad_trust', 'discount_importance', 'choose_promoted_product_likelihood'
]

# Impute using k=5 (all 5 features naturally share the 1-5 range)
knn = KNNImputer(n_neighbors=5, weights='distance')
imputed_matrix = np.round(knn.fit_transform(df_clean[knn_features]))
df_clean[knn_features] = imputed_matrix

# Restore human-readable categorical labels
rev_age_rank = {v: k for k, v in age_rank.items()}
rev_freq_rank = {v: k for k, v in freq_rank.items()}
df_clean['age_group'] = df_clean['age_group_ord'].map(rev_age_rank)
df_clean['ad_frequency'] = df_clean['ad_frequency_ord'].map(rev_freq_rank)

# Drop temporary ordinal helper columns
df_clean = df_clean.drop(columns=['age_group_ord', 'ad_frequency_ord'])
print("  ✓ Imputed numeric and ordinal features. Native whole numbers preserved.")

# ------------------------------------------------------------------------------
# 2-C. LOGICAL CONSISTENCY & NOMINAL MODE IMPUTATION
# ------------------------------------------------------------------------------
print("\n--- [STAGE 3] LOGICAL CONSISTENCY & MODE IMPUTATION ---")

nominal_cols = [
    'discovery_platform', 'online_product_category', 'clicked_ad_3m',
    'purchase_influence', 'ad_ignore_reason', 'preferred_promotion'
]
for col in nominal_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Enforce business logic: non-purchasers receive 'Not applicable'
df_clean.loc[df_clean['purchased_after_ad_3m'] == 'No', 'purchase_ad_platform'] = 'Not applicable'
df_clean['purchase_ad_platform'] = df_clean['purchase_ad_platform'].fillna('Not applicable')
print("  ✓ Mode imputation complete. Logical consistency verified.")

# ------------------------------------------------------------------------------
# 2-D. DOMAIN-AWARE SCALE INTEGRITY (OUTLIER VERIFICATION)
# ------------------------------------------------------------------------------
print("\n--- [STAGE 4] SCALE INTEGRITY VERIFICATION ---")

likert_cols = ['ad_trust', 'discount_importance', 'choose_promoted_product_likelihood']
for col in likert_cols:
    # Enforce strict survey boundary integrity [1, 5] as integers
    df_clean[col] = df_clean[col].clip(lower=1, upper=5).astype(int)
    print(f"  ✓ {col}: Min = {df_clean[col].min()}, Max = {df_clean[col].max()} (1-5 integer scale preserved)")

# ------------------------------------------------------------------------------
# 2-E. REDUNDANCY CLEANING & COMPOSITE SCORE (ON NATIVE 1-5 SCALE)
# ------------------------------------------------------------------------------
print("\n--- [STAGE 5] FINAL CLEANING & DIRECT COMPOSITE SCORING ---")

# Drop zero-variance administrative column
if 'status' in df_clean.columns:
    df_clean = df_clean.drop(columns=['status'])
    print("  ✓ Dropped uninformative constant column: 'status'")

# Composite consumer score on native 1.0 to 5.0 scale (no decimal scaling needed)
df_clean['ad_receptivity_score'] = df_clean[likert_cols].mean(axis=1).round(2)
print("  ✓ Created 'ad_receptivity_score' directly on native 1.0–5.0 scale.")

# ------------------------------------------------------------------------------
# 2-F. SAVE CLEANED DATASET
# ------------------------------------------------------------------------------
output_filename = "cleaned_data.csv"
df_clean.to_csv(output_filename, index=False)

print("\n" + "="*55)
print(f"PIPELINE COMPLETE! Saved to '{output_filename}'")
print(f"Cleaned Shape: {df_clean.shape[0]} Rows x {df_clean.shape[1]} Columns")
print(f"Total Missing Values Remaining: {df_clean.isnull().sum().sum()}")
print("="*55)

--- [STAGE 1] TEXT CLEANING & 1-5 ORDINAL ENCODING ---
  ✓ Converted demographics to balanced 1–5 ordinal scales.

--- [STAGE 2] 5D DISTANCE-WEIGHTED KNN IMPUTATION ---
  ✓ Imputed numeric and ordinal features. Native whole numbers preserved.

--- [STAGE 3] LOGICAL CONSISTENCY & MODE IMPUTATION ---
  ✓ Mode imputation complete. Logical consistency verified.

--- [STAGE 4] SCALE INTEGRITY VERIFICATION ---
  ✓ ad_trust: Min = 1, Max = 5 (1-5 integer scale preserved)
  ✓ discount_importance: Min = 1, Max = 5 (1-5 integer scale preserved)
  ✓ choose_promoted_product_likelihood: Min = 1, Max = 5 (1-5 integer scale preserved)

--- [STAGE 5] FINAL CLEANING & DIRECT COMPOSITE SCORING ---
  ✓ Dropped uninformative constant column: 'status'
  ✓ Created 'ad_receptivity_score' directly on native 1.0–5.0 scale.

PIPELINE COMPLETE! Saved to 'cleaned_data.csv'
Cleaned Shape: 80 Rows x 16 Columns
Total Missing Values Remaining: 0


Step 3: Download Cleaned Files

In [4]:
# ==============================================================================
# CELL 3 / STEP 3: DOWNLOAD CLEANED DATASET
# ==============================================================================
from google.colab import files

print("Downloading 'cleaned_data.csv' to your local machine...")
files.download("cleaned_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>